KHOI TAO


In [22]:
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader

from data import CatsDogsDataset, get_transforms
from utils import get_device, device_info
from evaluate import collect_predictions, compute_metrics

# Cấu hình
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = get_device()
print("Device:", device_info(DEVICE))

DATA_DIR = Path(r"datastet\raw")
OUTPUT_DIR = Path("outputs/metrics")

CLASS_NAMES = ["cats", "dogs"]
MODEL_NAMES = ["lenet", "alexnet", "resnet18", "vgg16"]

Device: CPU


Load 4 mô hình đã huấn luyện

In [23]:
from models.lenet import LeNet
from models.alexnet import AlexNet
from models.resnet18 import ResNet18
from models.vgg import VGG16

# Khởi tạo model
trained_lenet = LeNet()
trained_alexnet = AlexNet()
trained_resnet = ResNet18()
trained_vgg = VGG16()

# Đường dẫn checkpoint
CHECKPOINTS = {
    "lenet": OUTPUT_DIR / "lenet_best.pth",
    "alexnet": OUTPUT_DIR / "alexnet_best.pth",
    "resnet18": OUTPUT_DIR / "resnet18_best.pth",
    "vgg16": OUTPUT_DIR / "vgg16_best.pth",
}

# Dictionary lưu model
trained_map = {
    "lenet": trained_lenet,
    "alexnet": trained_alexnet,
    "resnet18": trained_resnet,
    "vgg16": trained_vgg,
}

# Load weights
for name, model in trained_map.items():
    checkpoint_path = CHECKPOINTS[name]

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Không tìm thấy checkpoint: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True
    )

    # Hỗ trợ cả checkpoint lưu trực tiếp state_dict
    # và checkpoint có chứa key model_state_dict
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()

    print(f"✓ Loaded {name.upper()}")

print("\nĐã load xong 4 mô hình!")

✓ Loaded LENET
✓ Loaded ALEXNET
✓ Loaded RESNET18
✓ Loaded VGG16

Đã load xong 4 mô hình!


Load tập Test

In [24]:
# Transform cho từng model
transforms_map = {
    "lenet": get_transforms("lenet", "test", augment=False),
    "alexnet": get_transforms("alexnet", "test", augment=False),
    "resnet18": get_transforms("resnet18", "test", augment=False),
    "vgg16": get_transforms("vgg16", "test", augment=False),
}

# Dataset Test
ds_test = CatsDogsDataset(DATA_DIR, "test")

print("Số lượng ảnh Test:", len(ds_test))
print("Số lượng cats:", sum(label == 0 for _, label in ds_test.samples))
print("Số lượng dogs:", sum(label == 1 for _, label in ds_test.samples))

Số lượng ảnh Test: 5000
Số lượng cats: 2500
Số lượng dogs: 2500


Dự đoán trên toàn bộ tập Test

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 0

results = {}

for name, model in trained_map.items():

    print(f"\nĐang đánh giá {name.upper()}...")

    # Dataset với transform tương ứng
    test_dataset = CatsDogsDataset(
        root=DATA_DIR,
        split="test",
        transform=transforms_map[name]
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    # Thu thập predictions
    labels, preds, probs = collect_predictions(
        model=model,
        loader=test_loader,
        device=DEVICE
    )

    # Tính metrics
    metrics = compute_metrics(
        labels=labels,
        preds=preds,
        class_names=CLASS_NAMES
    )

    results[name] = {
        "labels": labels,
        "preds": preds,
        "probs": probs,
        "metrics": metrics
    }

    print(
        f"{name.upper()} | "
        f"Accuracy: {metrics['accuracy']:.2f}% | "
        f"Macro F1: {metrics['macro_f1']:.2f}%"
    )

print("\nĐã đánh giá xong toàn bộ 4 mô hình!")


Đang đánh giá LENET...
LENET | Accuracy: 76.74% | Macro F1: 76.71%

Đang đánh giá ALEXNET...


Bảng so sánh tổng hợp 4 mô hình

In [ ]:
summary_rows = []

MODEL_DISPLAY_NAMES = {
    "lenet": "LeNet-5",
    "alexnet": "AlexNet",
    "resnet18": "ResNet-18",
    "vgg16": "VGG16"
}

for name, result in results.items():

    m = result["metrics"]

    summary_rows.append({
        "Metric": "Accuracy (%)",
        MODEL_DISPLAY_NAMES[name]: m["accuracy"]
    })

# Tạo bảng theo từng metric
metric_names = [
    "Accuracy (%)",
    "Macro Precision (%)",
    "Macro Recall (%)",
    "Macro F1 (%)",
    "N samples"
]

metric_keys = [
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "n_samples"
]

summary_data = []

for metric, key in zip(metric_names, metric_keys):

    row = {"Metric": metric}

    for name, result in results.items():
        row[MODEL_DISPLAY_NAMES[name]] = result["metrics"][key]

    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)

print("KẾT QUẢ SO SÁNH 4 MÔ HÌNH (TEST SET)")
display(
    df_summary.style
    .format({
        col: "{:.2f}" for col in df_summary.columns
        if col != "Metric"
    })
    .hide(axis="index")
)

Bảng kết quả theo từng class

In [ ]:
class_rows = []

for name, result in results.items():

    metrics = result["metrics"]

    for class_name in CLASS_NAMES:

        class_metric = metrics["per_class"][class_name]

        class_rows.append({
            "Model": MODEL_DISPLAY_NAMES[name],
            "Class": class_name,
            "Precision (%)": class_metric["precision"],
            "Recall (%)": class_metric["recall"],
            "F1-Score (%)": class_metric["f1"],
            "Support": class_metric["support"]
        })

df_class = pd.DataFrame(class_rows)

print("KẾT QUẢ THEO TỪNG CLASS (TEST SET)")

display(
    df_class.style
    .format({
        "Precision (%)": "{:.2f}",
        "Recall (%)": "{:.2f}",
        "F1-Score (%)": "{:.2f}",
        "Support": "{:,.0f}"
    })
    .hide(axis="index")
)

Confusion Matrix cho cả 4 mô hình

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

axes = axes.ravel()

for ax, (name, result) in zip(axes, results.items()):

    cm = result["metrics"]["confusion_matrix"]

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
        annot_kws={"size": 13}
    )

    ax.set_title(
        f"{MODEL_DISPLAY_NAMES[name]}\n"
        f"Accuracy: {result['metrics']['accuracy']:.2f}%",
        fontsize=13,
        fontweight="bold"
    )

    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label", fontsize=11)

plt.suptitle(
    "Confusion Matrix Comparison - Test Set",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

Hiển thị dự đoán chi tiết 10 ảnh Test

In [ ]:
# Lấy ngẫu nhiên 10 ảnh từ tập Test
sample_pairs = random.sample(
    ds_test.samples,
    min(10, len(ds_test.samples))
)

fig, axes = plt.subplots(2, 5, figsize=(20, 10))

for ax, (img_path, true_label) in zip(axes.ravel(), sample_pairs):

    img = Image.open(img_path).convert("RGB")

    ax.imshow(img)

    true_cls = CLASS_NAMES[true_label]
    preds_str = []

    for name, model in trained_map.items():

        tf = transforms_map[name]

        with torch.no_grad():

            tensor = tf(img).unsqueeze(0).to(DEVICE)

            logits = model(tensor)

            probs = torch.softmax(logits, dim=1)

            idx = probs.argmax(dim=1).item()

            confidence = probs[0, idx].item() * 100

        pred_cls = CLASS_NAMES[idx]

        status = "✓" if idx == true_label else "✗"

        preds_str.append(
            f"{name.upper()}: {pred_cls} "
            f"({confidence:.1f}%) {status}"
        )

    ax.set_title(
        f"True: {true_cls}\n" + "\n".join(preds_str),
        fontsize=8,
        fontweight="bold",
        color="darkgreen" if all(
            CLASS_NAMES[
                int(results[name]["probs"][
                    ds_test.samples.index((img_path, true_label))
                ].argmax())
            ] == true_cls
            for name in trained_map
        ) else "black"
    )

    ax.axis("off")

plt.suptitle(
    "Prediction Comparison - 4 CNN Models (10 Random Test Images)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()